# PathSnap Model Training & Benchmark: Histogram-Based Gradient Boosting (HistGradientBoosting)
**Course:** UE24CS352A - Machine Learning (PES University)  
**Team 6:** PathSnap Distributed Map-Matching  
**Model Name:** `hist_gradient_boosting`  

---
### Notebook Objectives
1. Inspect hyperparameters, training pipeline, and environment provenance.
2. Review 5-fold group-aware cross-validation on 80 training trips (16 trips/fold).
3. Evaluate held-out generalization on the 10 student-test trips.
4. Verify observation-level Top-1 accuracy, Macro F1, and categorical log loss.


In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image

PROJ_ROOT = Path("..").resolve()
MODEL_DIR = PROJ_ROOT / "results" / "run_01" / "hist_gradient_boosting"
sys.path.insert(0, str(PROJ_ROOT / "scripts"))

print("Model output directory:", MODEL_DIR)


Model output directory: C:\Users\joshwin\PESU\3RD YEAR\5th sem\ML\team_6\results\run_01\hist_gradient_boosting


## 1. Model Configuration & Provenance
Inspection of hyperparameters, random seeds, and package dependencies.


In [2]:
with open(MODEL_DIR / "configuration.json") as f:
    config = json.load(f)
with open(MODEL_DIR / "provenance.json") as f:
    prov = json.load(f)

print(f"Algorithm: {prov['model']} (Random Seed: {prov['seed']})")
print(f"Scikit-Learn Version: {prov['sklearn']} | Python: {prov['python']}")
print("\nKey Parameters:")
for k in sorted(config.keys())[:12]:
    print(f"  {k}: {config[k]}")


Algorithm: hist_gradient_boosting (Random Seed: 212)
Scikit-Learn Version: 1.9.1 | Python: 3.12.6

Key Parameters:
  categorical_features: from_dtype
  class_weight: None
  early_stopping: auto
  interaction_cst: None
  l2_regularization: 1.0
  learning_rate: 0.1
  loss: log_loss
  max_bins: 255
  max_depth: None
  max_features: 1.0
  max_iter: 150
  max_leaf_nodes: 31


## 2. 5-Fold Cross-Validation & Student-Test Evaluation
Group-aware 5-fold evaluation across all 80 training trips, followed by refitting on the full training set and testing on the 10 student-test trips.


In [3]:
with open(MODEL_DIR / "metrics.json") as f:
    records = json.load(f)

rows = []
for r in records:
    m = r["metrics"]["all_points"]
    rel = r["metrics"]["reliable_points"]
    rows.append({
        "Split": r["split"],
        "Top-1 Accuracy": f"{m['top1_accuracy']*100:.2f}%",
        "Macro F1": f"{m['macro_f1']:.4f}",
        "Macro Precision": f"{m['macro_precision']:.4f}",
        "Macro Recall": f"{m['macro_recall']:.4f}",
        "Log Loss": f"{m['categorical_log_loss_with_miss_state']:.4f}",
        "Reliable Top-1": f"{rel['top1_accuracy']*100:.2f}%",
        "Fit Time (s)": f"{r['fit_seconds']:.1f}s",
        "Inf Time (ms)": f"{r['inference_seconds']*1000:.1f}ms"
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics)


,Split,Top-1 Accuracy,Macro F1,Macro Precision,Macro Recall,Log Loss,Reliable Top-1,Fit Time (s),Inf Time (ms)
0,fold_0,77.35%,0.6062,0.6442,0.5898,0.9564,86.17%,9.0s,611.2ms
1,fold_1,75.24%,0.6854,0.7168,0.6721,3.0927,87.81%,8.2s,766.8ms
2,fold_2,46.49%,0.5887,0.6311,0.5693,7.9047,81.29%,8.6s,906.5ms
3,fold_3,75.19%,0.6515,0.6907,0.6319,0.7804,77.95%,6.9s,1130.9ms
4,fold_4,84.89%,0.7034,0.7347,0.6913,0.6092,91.73%,8.7s,548.3ms
5,student_test,80.78%,0.6113,0.6530,0.5959,0.6710,89.01%,9.9s,375.6ms


## 3. Performance Summary
Mean cross-validation metrics across all 5 folds.


In [4]:
with open(MODEL_DIR / "summary.json") as f:
    summary = json.load(f)

print(f"Completed Folds: {summary['completed_folds']}")
print(f"Mean CV Macro F1: {summary['cv_macro_f1_mean']:.4f} ± {summary['cv_macro_f1_std']:.4f}")
print(f"Mean CV Top-1 Accuracy: {summary['cv_top1_acc_mean']*100:.2f}%")
print(f"Student-Test Evaluation Complete: {summary['student_test_completed']}")


Completed Folds: 5
Mean CV Macro F1: 0.6471 ± 0.0441
Mean CV Top-1 Accuracy: 71.83%
Student-Test Evaluation Complete: True
